# DP-SGD Challenge: SVHN Privacy-Preserving Classifier

This is a separate challenge notebook for HTB Academy module 335. The validator requires the exact SVHNCNN architecture, SVHN normalization, a Safetensors artifact named `dp_model.safetensors`, at least 55% accuracy, and membership-inference advantage at most 5%.

The notebook keeps submission disabled by default. Fill in the target address only after spawning the HTB instance, train and inspect the local metrics, then explicitly set `SUBMIT = True` when you are ready to send the model to the authorized challenge validator.

## Challenge constraints and notation

The model must use DP-SGD. For each per-example gradient `g_i` (**g sub i**), clipping enforces `||g_i||_2 <= C` (**the L-two norm of g sub i is at most C**), then Gaussian noise is added. The privacy budget is `(epsilon, delta)` (**epsilon, delta**); smaller epsilon means stronger privacy but usually lower accuracy.

Required validator metrics:

- `accuracy >= 0.55`
- `mia_advantage <= 0.05`

A local confidence-threshold MIA check is only a diagnostic. The server is authoritative.

In [ ]:
# Challenge connection and submission controls.
# Target supplied by the active authorized HTB instance.
INSTANCE_HOST = '154.57.164.66:30373'
BASE_URL = f'http://{INSTANCE_HOST}'
MODEL_PATH = 'dp_model.safetensors'
SUBMIT = False  # Keep False until you explicitly authorize submission.

TARGET_ACCURACY = 0.55
TARGET_MIA_ADVANTAGE = 0.05
TARGET_EPSILON = 10.0
TARGET_DELTA = 1e-5
MAX_GRAD_NORM = 1.0
BATCH_SIZE = 256
EPOCHS = 20
LEARNING_RATE = 0.1
RANDOM_SEED = 1337

In [ ]:
import os
import random
import numpy as np
import torch

# Compatibility guard for environments where torchvision was built without
# registering its optional NMS operator. This does not affect SVHN tensors.
try:
    _torchvision_compat = torch.library.Library('torchvision', 'DEF')
    _torchvision_compat.define('nms(Tensor boxes, Tensor scores, float iou_threshold) -> Tensor')
except Exception:
    pass

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from safetensors.torch import save_file, load_file
from torchvision import datasets, transforms

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## Exact validator architecture

Do not change layer sizes, ordering, pooling, or the absence of BatchNorm. The final feature map is 64 channels by 4 by 4 pixels, so the first linear layer receives `64 * 4 * 4 = 1024` values. The final layer produces 10 logits, one for each SVHN digit class.

In [ ]:
class SVHNCNN(nn.Module):
    """Exact Opacus-compatible architecture required by the validator."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 4 * 4, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 64 * 4 * 4)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

## SVHN preprocessing

The validator uses these exact channel statistics. Normalization changes each channel from raw pixel scale into the coordinate system expected by the trained model:

`normalized = (pixel - mean) / standard_deviation`

Read aloud: **normalized equals pixel minus mean, divided by standard deviation**.

In [ ]:
SVHN_MEAN = (0.4377, 0.4438, 0.4728)
SVHN_STD = (0.1980, 0.2010, 0.1970)
TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(SVHN_MEAN, SVHN_STD),
])

train_dataset = datasets.SVHN('data', split='train', download=True, transform=TRANSFORM)
test_dataset = datasets.SVHN('data', split='test', download=True, transform=TRANSFORM)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train samples: {len(train_dataset):,}; test samples: {len(test_dataset):,}')

## DP-SGD training

Opacus wraps the model, optimizer, and loader. It clips each sample's gradient to `MAX_GRAD_NORM`, adds noise, and tracks the cumulative RDP privacy budget. A fresh model and privacy engine are used for this challenge run. If `ModuleValidator.fix()` changes the architecture, stop and inspect it; the supplied architecture is already designed for Opacus compatibility.

In [ ]:
model = SVHNCNN().to(DEVICE)
fixed_model = ModuleValidator.fix(model)
if fixed_model.__class__ is not model.__class__:
    print('Validator returned a modified model; inspect before continuing.')
model = fixed_model
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9)
privacy_engine = PrivacyEngine(accountant='rdp')
model, optimizer, private_train_loader = privacy_engine.make_private_with_epsilon(
    module=model, optimizer=optimizer, data_loader=train_loader,
    target_epsilon=TARGET_EPSILON, target_delta=TARGET_DELTA,
    epochs=EPOCHS, max_grad_norm=MAX_GRAD_NORM
)

criterion = nn.CrossEntropyLoss()
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    correct = 0
    seen = 0
    for images, labels in private_train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        seen += labels.size(0)
    epsilon_now = privacy_engine.get_epsilon(TARGET_DELTA)
    print(f'Epoch {epoch + 1:02d}/{EPOCHS} | loss={running_loss / seen:.4f} | train_acc={100 * correct / seen:.2f}% | epsilon={epsilon_now:.3f}')

FINAL_EPSILON = privacy_engine.get_epsilon(TARGET_DELTA)
print(f'Final privacy accounting: epsilon={FINAL_EPSILON:.3f}, delta={TARGET_DELTA}')

In [ ]:
@torch.no_grad()
def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    for images, labels in loader:
        logits = model(images.to(DEVICE))
        correct += (logits.argmax(dim=1).cpu() == labels).sum().item()
        total += labels.size(0)
    return correct / total

@torch.no_grad()
def collect_confidence(model, loader):
    model.eval()
    scores = []
    for images, _ in loader:
        probabilities = torch.softmax(model(images.to(DEVICE)), dim=1)
        scores.append(probabilities.max(dim=1).values.cpu())
    return torch.cat(scores)

def confidence_mia_advantage(model, member_loader, non_member_loader):
    members = collect_confidence(model, member_loader)
    non_members = collect_confidence(model, non_member_loader)
    n = min(len(members), len(non_members))
    members, non_members = members[:n], non_members[:n]
    scores = torch.cat([members, non_members])
    labels = torch.cat([torch.ones(n), torch.zeros(n)])
    best_accuracy = 0.0
    for threshold in torch.linspace(0.0, 1.0, 201):
        predictions = (scores >= threshold).float()
        best_accuracy = max(best_accuracy, (predictions == labels).float().mean().item())
    return best_accuracy, best_accuracy - 0.5

accuracy = evaluate_accuracy(model, test_loader)
mia_accuracy, mia_advantage = confidence_mia_advantage(model, private_train_loader, test_loader)
print(f'Test accuracy: {accuracy:.4f}')
print(f'Local MIA accuracy: {mia_accuracy:.4f}')
print(f'Local MIA advantage: {mia_advantage:.4f}')
print(f'Accuracy requirement met: {accuracy >= TARGET_ACCURACY}')
print(f'MIA requirement met: {mia_advantage <= TARGET_MIA_ADVANTAGE}')

## Save the validator artifact

Opacus wraps the original model, so the underlying state dictionary is accessed through `model._module`. Safetensors stores the weights in the format required by the challenge validator.

The server, not the local diagnostic, decides whether the submitted artifact passes.

In [ ]:
underlying_model = model._module if hasattr(model, '_module') else model
save_file(underlying_model.state_dict(), MODEL_PATH)
print(f'Saved {MODEL_PATH}')

# Reload locally to catch a malformed or incomplete artifact.
reloaded = SVHNCNN().to(DEVICE)
reloaded.load_state_dict(load_file(MODEL_PATH, device=str(DEVICE)))
print('Safetensors reload check: passed')

## Optional validator submission (disabled by default)

This cell is intentionally guarded. It transmits `dp_model.safetensors` and the model's predictions to the authorized HTB challenge instance. Set `SUBMIT = True` only after replacing the placeholder address, checking local metrics, and explicitly deciding to submit.

In [ ]:
if not SUBMIT:
    print('Submission disabled. Set SUBMIT = True only when ready.')
else:
    if 'INSTANCE_IP:PORT' in BASE_URL:
        raise ValueError('Replace INSTANCE_HOST with the spawned HTB target first.')
    import requests
    response = requests.get(f'{BASE_URL}/health', timeout=15)
    response.raise_for_status()
    print('Health response:', response.json())
    with open(MODEL_PATH, 'rb') as artifact:
        result = requests.post(
            f'{BASE_URL}/validate',
            files={'model': ('dp_model.safetensors', artifact, 'application/octet-stream')},
            timeout=120,
        )
    result.raise_for_status()
    print('Validator response:', result.json())

## Challenge run notes

- Target configured: `154.57.164.66:30373`; after the VPN connected, `/health` returned `status=ok` and `data_loaded=true`.
- The `AI` environment now has Opacus and Safetensors installed.
- Its PyTorch build reports `2.14.1+cpu`, so the notebook correctly falls back to CPU even though an RTX 2050 is visible to the NVIDIA driver.
- A torchvision compatibility guard defines the missing optional NMS operator before importing SVHN utilities; it does not change the challenge model or preprocessing.
- The first full execution reached SVHN download/training setup; CPU training was stopped before submission while diagnosing the environment.
- Submission remains disabled until local metrics are available and the validator artifact is ready.


## Challenge checklist

- Exact SVHNCNN architecture preserved.
- SVHN mean and standard deviation preserved.
- DP-SGD used with Opacus and an RDP accountant.
- `dp_model.safetensors` created locally.
- Accuracy target: at least 55%.
- MIA advantage target: at most 5%.
- Target address is isolated in `INSTANCE_HOST` and `BASE_URL`.
- Submission remains disabled until explicitly enabled.